# FunctionGemma: a first look

**Goal:** see what an on-device function-calling model does *out of the box*, before we fine-tune it.

Model: `google/functiongemma-270m-it` (270M parameters, Gemma 3 270M fine-tuned for function calling, released Dec 2025)


In [ ]:
%%capture
!pip install -U transformers accelerate

## 1. Load the model


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# If the Google repo asks you to accept the Gemma license, either accept it on the
# Hugging Face page or use the mirror: "unsloth/functiongemma-270m-it"
MODEL_ID = "google/functiongemma-270m-it"

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto").to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2
print(f"device: {device} | dtype: {model.dtype}")
print(f"parameters: {n_params / 1e6:.0f}M | weights in memory: {size_mb:.0f} MB")

/home/lpozzi/Git/data-science-lectures/llm_lora/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/lpozzi/Git/data-science-lectures/llm_lora/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


device: cuda | dtype: torch.bfloat16
parameters: 268M | weights in memory: 511 MB


In [2]:
print(model)

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 640, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=640, out_features=1024, bias=False)
          (k_proj): Linear(in_features=640, out_features=256, bias=False)
          (v_proj): Linear(in_features=640, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=640, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=640, out_features=2048, bias=False)
          (up_proj): Linear(in_features=640, out_features=2048, bias=False)
          (down_proj): Linear(in_features=2048, out_features=640, bias=False)
          (act_fn): GELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((640,), eps=1e-06)

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "set_alarm",
            "description": "Sets an alarm on the phone at a given time.",
            "parameters": {
                "type": "object",
                "properties": {
                    "time": {"type": "string", "description": "Alarm time in HH:MM 24h format"},
                    "label": {"type": "string", "description": "Optional label for the alarm"},
                },
                "required": ["time"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "send_message",
            "description": "Sends a text message to a contact.",
            "parameters": {
                "type": "object",
                "properties": {
                    "contact": {"type": "string", "description": "Name of the contact"},
                    "text": {"type": "string", "description": "Body of the message"},
                },
                "required": ["contact", "text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "create_calendar_event",
            "description": "Creates an event in the calendar.",
            "parameters": {
                "type": "object",
                "properties": {
                    "title": {"type": "string", "description": "Title of the event"},
                    "date": {"type": "string", "description": "Date in YYYY-MM-DD format"},
                    "time": {"type": "string", "description": "Start time in HH:MM 24h format"},
                },
                "required": ["title", "date"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "toggle_setting",
            "description": "Turns a phone setting on or off.",
            "parameters": {
                "type": "object",
                "properties": {
                    "setting": {
                        "type": "string",
                        "enum": ["wifi", "bluetooth", "flashlight", "do_not_disturb"],
                        "description": "The setting to change",
                    },
                    "enabled": {"type": "boolean", "description": "true to turn on, false to turn off"},
                },
                "required": ["setting", "enabled"],
            },
        },
    },
]

## 3. What does the model actually see?
FunctionGemma needs a **developer message** that activates function calling.
The chat template injects the tool schemas into the prompt.
How many tokens do the tool definitions cost? On a phone, every prompt token is latency.

In [4]:
DEVELOPER_PROMPT = "You are a model that can do function calling with the following functions"

def build_messages(user_request):
    return [
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": user_request},
    ]

prompt_text = tokenizer.apply_chat_template(
    build_messages("Set an alarm for 7am"),
    tools=TOOLS,
    add_generation_prompt=True,
    tokenize=False,
)
print(prompt_text)
print("prompt tokens:", len(tokenizer(prompt_text, add_special_tokens=False)["input_ids"]))

<bos><start_of_turn>developer
You are a model that can do function calling with the following functions<start_function_declaration>declaration:set_alarm{description:<escape>Sets an alarm on the phone at a given time.<escape>,parameters:{properties:{label:{description:<escape>Optional label for the alarm<escape>,type:<escape>STRING<escape>},time:{description:<escape>Alarm time in HH:MM 24h format<escape>,type:<escape>STRING<escape>}},required:[<escape>time<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:send_message{description:<escape>Sends a text message to a contact.<escape>,parameters:{properties:{contact:{description:<escape>Name of the contact<escape>,type:<escape>STRING<escape>},text:{description:<escape>Body of the message<escape>,type:<escape>STRING<escape>}},required:[<escape>contact<escape>,<escape>text<escape>],type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:create_calendar

## 4. Generate and parse a function call
The model does **not** return JSON. It emits its own format:
```
<start_function_call>call:set_alarm{time:<escape>07:00<escape>}<end_function_call>
```
Parsing it into a Python call is the app's job.

In [5]:
import re
import json
import time

CALL_RE = re.compile(r"<start_function_call>call:([\w\.]+)\{(.*?)\}<end_function_call>", re.S)
ARG_RE = re.compile(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]+))", re.S)


def parse_value(raw):
    raw = raw.strip()
    try:
        return json.loads(raw)  # numbers, true/false, null
    except json.JSONDecodeError:
        return raw


def parse_calls(text):
    """Extract [{"name": ..., "arguments": {...}}] from FunctionGemma output.
    Flat arguments only: nested objects/arrays are not handled."""
    calls = []
    for name, args_str in CALL_RE.findall(text):
        args = {}
        for key, escaped, bare in ARG_RE.findall(args_str):
            args[key] = escaped if escaped or not bare else parse_value(bare)
        calls.append({"name": name, "arguments": args})
    return calls


@torch.no_grad()
def ask(user_request, max_new_tokens=128):
    inputs = tokenizer.apply_chat_template(
        build_messages(user_request),
        tools=TOOLS,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)
    start = time.perf_counter()
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # deterministic: same request, same call
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.perf_counter() - start
    new_tokens = out[0][inputs["input_ids"].shape[-1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return {"raw": raw, "calls": parse_calls(raw), "seconds": elapsed, "tokens": len(new_tokens)}


r = ask("Set an alarm for 7am")
print(r["raw"])
print(r["calls"])
print(f"{r['tokens']} tokens in {r['seconds']:.2f}s")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<start_function_call>call:set_alarm{time:<escape>07:00<escape>}<end_function_call><start_function_response>
[{'name': 'set_alarm', 'arguments': {'time': '07:00'}}]
19 tokens in 1.56s


# The task: a voice command layer for a mobile app

## The product
Most mobile apps expose a set of actions: create, edit, search, delete, change settings...
Today the user reaches them by tapping through menus.

We want to add a **voice layer**: the user speaks a command in natural language,
and the app executes the corresponding action.

> 🎙️ "Crea un promemoria per domani alle 9"
> → the app calls `create_reminder(...)`

The language model does not chat and does not answer questions:
its only job is to translate a spoken request into a **call to one of the app's functions**.

## Why on-device?
- **Privacy**: the user's commands and data never leave the phone
- **Latency**: a command must feel instant (< 1 s)
- **Offline**: works without a connection
- **Cost**: zero inference cost per command, whatever the number of users

## Architecture

```
 🎙️ voice
   │
   ▼
 ┌──────────────────┐
 │ ASR (on-device)  │  speech → text
 └────────┬─────────┘
          ▼
 ┌──────────────────┐
 │ FunctionGemma    │  text → function call
 │ 270M + LoRA      │
 └────────┬─────────┘
          ▼
 ┌──────────────────┐
 │ Validator        │  schema check, resolve references,
 │ (plain code)     │  confirmation for destructive actions
 └────────┬─────────┘
          ▼
 ┌──────────────────┐        ┌───────────────────────┐
 │ App functions    │        │ Cloud / bigger model  │
 │ (executor)       │        │ for requests too      │
 └──────────────────┘        │ complex for on-device │
                             └───────────────────────┘
```

**Today's focus: the FunctionGemma box**, and how to make a 270M model reliable at it.

## Design principle: the model extracts, the app resolves

A 270M model is a specialist. We keep its job as small as possible:

| the model does | the app does (deterministic code) |
|---|---|
| choose the function | match names to real objects in the app (fuzzy search) |
| copy names and values as the user said them | turn "domani" or "la settimana scorsa" into real dates |
| map vague expressions to fixed values (enums) | check that values are valid |
| use placeholders for context ("questo", "quello che ho copiato") | know what is on screen or in the clipboard |
| decide that *no* function applies | handle the conversation around it |

Every decision we move into code is accuracy we don't need to train.

## Voice changes the input
The model never sees typed text, it sees **transcripts**:
- no punctuation, often lowercase: `crea un promemoria domani alle nove`
- numbers as words: `nove e mezza`, `dodici euro e cinquanta`
- hesitations and restarts: `ehm crea no anzi cerca le note su...`
- ASR mistakes on names and technical terms: `lora` → `l'ora`

Our dataset will have to look like this, not like clean written text.

## The use case: a voice layer for a notes app (Zettelkasten)

The app manages a personal knowledge base of atomic notes, organized in folders,
connected by links and tags. Through voice the user can:
create, append to, link, tag, search, move and delete notes,
and send a video to the cloud pipeline that turns it into notes.

Two placeholders let the model refer to context it cannot see:
- `@current` → the note currently open on screen
- `@clipboard` → the link the user has copied

In [6]:
FOLDERS = ["inbox", "fleeting", "literature", "permanent"]
PERIODS = ["today", "yesterday", "this_week", "last_week", "this_month"]

def tool(name, description, properties, required):
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {"type": "object", "properties": properties, "required": required},
        },
    }

NOTE_REF = {"type": "string",
            "description": "Title of the note as said by the user, or '@current' for the note currently open"}

NOTE_TOOLS = [
    tool("create_note", "Creates a new note.", {
        "title": {"type": "string", "description": "Title of the note"},
        "content": {"type": "string", "description": "Optional initial text of the note"},
        "tags": {"type": "string", "description": "Optional comma-separated tags, e.g. 'nlp,lora'"},
        "folder": {"type": "string", "enum": FOLDERS, "description": "Folder where the note is created"},
    }, ["title"]),
    tool("append_to_note", "Appends text at the end of an existing note.", {
        "note": NOTE_REF,
        "text": {"type": "string", "description": "Text to append"},
    }, ["note", "text"]),
    tool("link_notes", "Creates a link from one note to another.", {
        "source": NOTE_REF,
        "target": {"type": "string", "description": "Title of the note to link to"},
    }, ["source", "target"]),
    tool("add_tags", "Adds one or more tags to a note.", {
        "note": NOTE_REF,
        "tags": {"type": "string", "description": "Comma-separated tags, e.g. 'nlp,lora'"},
    }, ["note", "tags"]),
    tool("search_notes", "Searches notes by text, tag or creation period.", {
        "query": {"type": "string", "description": "Words to search for"},
        "tag": {"type": "string", "description": "Only notes with this tag"},
        "period": {"type": "string", "enum": PERIODS, "description": "Only notes created in this period"},
    }, []),
    tool("move_note", "Moves a note to another folder.", {
        "note": NOTE_REF,
        "folder": {"type": "string", "enum": FOLDERS, "description": "Destination folder"},
    }, ["note", "folder"]),
    tool("delete_note", "Deletes a note.", {
        "note": NOTE_REF,
    }, ["note"]),
    tool("ingest_video", "Sends a video to be transformed into atomic notes.", {
        "source": {"type": "string",
                   "description": "Video URL, or '@clipboard' if the user refers to a copied link"},
    }, ["source"]),
]

print([t["function"]["name"] for t in NOTE_TOOLS])

['create_note', 'append_to_note', 'link_notes', 'add_tags', 'search_notes', 'move_note', 'delete_note', 'ingest_video']


In [7]:
# Inputs look like ASR transcripts: lowercase, no punctuation.
# expected = [] means the model should NOT call any function.
def call(name, **arguments):
    return {"name": name, "arguments": arguments}

PROBES = [
    # --- basic: one function, one argument
    ("basic_en", "create a note called attention mechanisms",
     [call("create_note", title="attention mechanisms")]),
    ("basic_it", "crea una nota sulla lora",
     [call("create_note", title="lora")]),
    ("basic_it", "elimina la nota su bert",
     [call("delete_note", note="bert")]),

    # --- several arguments in one sentence
    ("multi_arg", "crea una nota su rag nella cartella permanent con i tag nlp e retrieval",
     [call("create_note", title="rag", folder="permanent", tags="nlp,retrieval")]),
    ("multi_arg", "aggiungi alla nota su vllm che supporta il multi lora",
     [call("append_to_note", note="vllm", text="supporta il multi lora")]),
    ("multi_arg", "collega la nota su lora a quella sulla quantizzazione",
     [call("link_notes", source="lora", target="quantizzazione")]),

    # --- vague expression -> enum value
    ("enum", "sposta la nota su transformers tra le note permanenti",
     [call("move_note", note="transformers", folder="permanent")]),
    ("enum", "metti la nota su whisper nella posta in arrivo",
     [call("move_note", note="whisper", folder="inbox")]),
    ("enum", "cerca le note di ieri sul fine tuning",
     [call("search_notes", query="fine tuning", period="yesterday")]),
    ("enum", "fammi vedere le note della settimana scorsa",
     [call("search_notes", period="last_week")]),

    # --- placeholders
    ("placeholder", "aggiungi il tag da rivedere a questa nota",
     [call("add_tags", note="@current", tags="da rivedere")]),
    ("placeholder", "collega questa nota a quella su attention",
     [call("link_notes", source="@current", target="attention")]),
    ("placeholder", "fai le note del video che ho copiato",
     [call("ingest_video", source="@clipboard")]),

    # --- ASR-style noise
    ("asr_noise", "ehm crea una nota no anzi cerca le note su transformers",
     [call("search_notes", query="transformers")]),
    ("asr_noise", "allora cerca tutte le note con il tag nlp grazie",
     [call("search_notes", tag="nlp")]),

    # --- required argument missing: should not invent one
    ("missing_arg", "sposta la nota",
     []),
    ("missing_arg", "aggiungi un tag alla nota su bert",
     []),

    # --- no function applies
    ("negative", "che tempo fa domani",
     []),
    ("negative", "grazie mille",
     []),
    ("negative", "cos'è la quantizzazione",
     []),
]

print(f"{len(PROBES)} probes")

20 probes


In [8]:
import pandas as pd

def ask_with(tools, user_request, max_new_tokens=128):
    messages = [
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": user_request},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True,
        return_dict=True, return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.eos_token_id)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=False)
    return raw, parse_calls(raw)


def norm(v):
    """Light normalization: free text is compared case-insensitively, tags as sets."""
    if isinstance(v, str):
        return v.strip().lower()
    return v

def same_args(pred, gold):
    if set(pred) != set(gold):
        return False
    for k in gold:
        if k == "tags":
            if {t.strip().lower() for t in str(pred[k]).split(",")} != {t.strip().lower() for t in gold[k].split(",")}:
                return False
        elif norm(pred[k]) != norm(gold[k]):
            return False
    return True


rows = []
for category, request, expected in PROBES:
    raw, calls = ask_with(NOTE_TOOLS, request)
    if not expected:
        correct_fn = exact = (len(calls) == 0)
    else:
        correct_fn = len(calls) == 1 and calls[0]["name"] == expected[0]["name"]
        exact = correct_fn and same_args(calls[0]["arguments"], expected[0]["arguments"])
    rows.append({
        "category": category,
        "request": request,
        "expected": expected,
        "predicted": calls if calls else raw.strip()[:60],
        "function_ok": correct_fn,
        "exact_match": exact,
    })

results = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
results

,category,request,expected,predicted,function_ok,exact_match
0,basic_en,create a note called attention mechanisms,"[{'name': 'create_note', 'arguments': {'title': 'attention mechanisms'}}]","[{'name': 'create_note', 'arguments': {'folder': 'literature', 'title': 'att...",True,False
1,basic_it,crea una nota sulla lora,"[{'name': 'create_note', 'arguments': {'title': 'lora'}}]","[{'name': 'create_note', 'arguments': {'folder': 'literature', 'title': 'lor...",True,False
2,basic_it,elimina la nota su bert,"[{'name': 'delete_note', 'arguments': {'note': 'bert'}}]","I apologize, but I cannot assist with deleting notes. My cur",False,False
3,multi_arg,crea una nota su rag nella cartella permanent con i tag nlp e retrieval,"[{'name': 'create_note', 'arguments': {'title': 'rag', 'folder': 'permanent'...","[{'name': 'create_note', 'arguments': {'content': 'New Note: Write to perman...",True,False
4,multi_arg,aggiungi alla nota su vllm che supporta il multi lora,"[{'name': 'append_to_note', 'arguments': {'note': 'vllm', 'text': 'supporta ...","[{'name': 'create_note', 'arguments': {'content': 'In this note, we will dis...",False,False
5,multi_arg,collega la nota su lora a quella sulla quantizzazione,"[{'name': 'link_notes', 'arguments': {'source': 'lora', 'target': 'quantizza...","[{'name': 'create_note', 'arguments': {'folder': 'literature', 'title': 'col...",False,False
6,enum,sposta la nota su transformers tra le note permanenti,"[{'name': 'move_note', 'arguments': {'note': 'transformers', 'folder': 'perm...","[{'name': 'move_note', 'arguments': {'folder': 'literature', 'note': 'perman...",True,False
7,enum,metti la nota su whisper nella posta in arrivo,"[{'name': 'move_note', 'arguments': {'note': 'whisper', 'folder': 'inbox'}}]",I cannot assist with writing or retrieving specific text con,False,False
8,enum,cerca le note di ieri sul fine tuning,"[{'name': 'search_notes', 'arguments': {'query': 'fine tuning', 'period': 'y...","[{'name': 'ingest_video', 'arguments': {'source': 'clipboard'}}]",False,False
9,enum,fammi vedere le note della settimana scorsa,"[{'name': 'search_notes', 'arguments': {'period': 'last_week'}}]","[{'name': 'search_notes', 'arguments': {'query': 'weekends'}}]",True,False


In [10]:
summary = (results.groupby("category")[["function_ok", "exact_match"]]
           .mean().mul(100).round(0)
           .assign(n=results.groupby("category").size()))
print(summary)
print(f"\nOVERALL  function: {results.function_ok.mean():.0%}   exact match: {results.exact_match.mean():.0%}")

             function_ok  exact_match  n
category                                
asr_noise           50.0          0.0  2
basic_en           100.0          0.0  1
basic_it            50.0          0.0  2
enum                50.0          0.0  4
missing_arg         50.0         50.0  2
multi_arg           33.0          0.0  3
negative           100.0        100.0  3
placeholder         67.0          0.0  3

OVERALL  function: 60%   exact match: 20%


## Evaluation taxonomy (BFCL-style)

We follow the taxonomy of the **Berkeley Function Calling Leaderboard (BFCL)**:
[BFCL v3 on EvalScope](https://evalscope.readthedocs.io/en/latest/third_party/bfcl_v3.html)

BFCL splits its single-turn data into *non-live* (curated by the authors) and *live* (real user queries)
subsets. We merge them: what matters for us is the **type of behavior** being tested, not where the query comes from.
We also leave out the **multi-turn** subsets (stateful conversations over several turns) and the
**Java/JavaScript** subsets (calls in other languages' syntax): our voice layer handles one command at a time,
in FunctionGemma's own format.

| category | what is given to the model | expected behavior | example in our app |
|---|---|---|---|
| **simple** | one function | one call, correct arguments | only `create_note` available: *"crea una nota sulla lora"* → `create_note(title="lora")` |
| **multiple** | several functions | choose the right one, one call | all tools: *"collega la nota su lora a quella sulla quantizzazione"* → `link_notes(source="lora", target="quantizzazione")` |
| **parallel** | one function | several calls of it in one turn | *"tagga nlp le note su bert e lora"* → `add_tags(note="bert", tags="nlp")`, `add_tags(note="lora", tags="nlp")` |
| **parallel_multiple** | several functions | several different calls in one turn | *"crea una nota su rag e collegala a retrieval"* → `create_note(title="rag")`, `link_notes(source="rag", target="retrieval")` |
| **irrelevance** | functions that don't fit the request | **no call** | *"che tempo fa domani"* → no call |
| **relevance** | at least one function fits | **some call** (not a refusal or a text answer) | *"elimina la nota su bert"* → a call, not *"I cannot assist…"* |

### How each category is scored
- **simple, multiple, parallel, parallel_multiple**: AST matching. The call is parsed and compared with the expected one:
  function name, parameter names, types, and values (strings compared case-insensitively, whitespace normalized).
  For parallel categories, every expected call must be present.
- **irrelevance**: correct if the model makes no call.
- **relevance**: correct if the model makes a call.

> ⚠️ **irrelevance** and **relevance** pull in opposite directions: a model that never calls a function
> scores 100% on irrelevance and 0% on relevance. Always read them together.

![](https://sail-moe.oss-cn-hangzhou.aliyuncs.com/yunlin/images/evalscope/doc/bfcl_v3.jpg)

In [2]:
import asyncio
from ragas.metrics.collections import ToolCallAccuracy
from ragas.messages import AIMessage, HumanMessage, ToolCall

async def evaluate_tool_call_accuracy():
    # Define the conversation with tool calls
    user_input = [
        HumanMessage(content="What's the weather like in New York right now?"),
        AIMessage(
            content="The current temperature in New York is 75°F and it's partly cloudy.",
            tool_calls=[ToolCall(name="weather_check", args={"location": "New York"})],
        ),
        HumanMessage(content="Can you translate that to Celsius?"),
        AIMessage(
            content="Let me convert that to Celsius for you.",
            tool_calls=[
                ToolCall(
                    name="temperature_conversion", args={"temperature_fahrenheit": 75}
                )
            ],
        ),
    ]

    # Define expected tool calls
    reference_tool_calls = [
        ToolCall(name="weather_check", args={"location": "New York"}),
        ToolCall(name="temperature_conversion", args={"temperature_fahrenheit": 75}),
    ]

    # Evaluate
    metric = ToolCallAccuracy()
    result = await metric.ascore(
        user_input=user_input,
        reference_tool_calls=reference_tool_calls,
    )
    print(f"Tool Call Accuracy: {result.value}")

await evaluate_tool_call_accuracy()

Tool Call Accuracy: 1.0


Scoring Examples
Perfect match:

```
# All tools called correctly with correct arguments
Expected: [weather_check(location="Paris"), translate(text="hello")]
Got:      [weather_check(location="Paris"), translate(text="hello")]
Score: 1.0
```

Partial argument match:

```
# Some arguments incorrect
Expected: [search(query="python", limit=10, sort="date")]
Got:      [search(query="python", limit=10, sort="relevance")]
Score: 0.66 (2 out of 3 arguments match)
```

Wrong order (strict mode):

```
# Correct tools but wrong sequence
Expected: [search(...), filter(...)]
Got:      [filter(...), search(...)]
Score: 0.0 (sequence not aligned)
```